# Google Trends, HCUP, & U.S. Census Bureau Data Collection
**Author:** J. Casey Brookshier  
**Date:** July 15, 2026

## Objective

Collect, clean, and merge Google Trends, HCUP, and U.S. Census Bureau data to build a state-year dataset for predicting future inpatient psychiatric admissions.

### Data Sources
**Google Trends:** State-level suicide/crisis search interest (predictors)
  **HCUP:** Annual Mental Health/Substance Use inpatient admissions (outcome)
  **U.S. Census Bureau:** State population estimates (normalization)

### Workflow
**Collect → Clean → Merge → Model → Evaluate**

The resulting dataset supports regression, Random Forest, XGBoost, PCA, hierarchical clustering, and SHAP feature importance analyses.

In [ ]:
# imports & project paths

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().resolve().parents[1]

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

In [ ]:
#define input/output files

GT_FILE = RAW_DIR / "google_trends_state_year_2013_2023.csv"
GT_RAW_FILE = RAW_DIR / "google_trends_raw_progress.csv"

HCUP_FILE = RAW_DIR / "hcup_state_year_mental_health_admissions_2013_2023.csv"
CENSUS_FILE = RAW_DIR / "uscb_state_population_2013_2023.csv"

OUTPUT_FILE = PROCESSED_DIR / "team_rho_state_year_prediction_dataset.csv"

SEARCH_TERMS = [
    "suicidal thoughts",
    "suicide hotline",
    "self harm",
    "mental health crisis",
    "suicide prevention",
    "crisis hotline",
    "psychiatric hospital",
    "depression help",
]

YEARS = range(2013, 2024)

In [ ]:
# verify repo structure

required_dirs = [
    RAW_DIR,
    PROCESSED_DIR,
]

for directory in required_dirs:
    if not directory.exists():
        raise FileNotFoundError(f"Missing directory: {directory}")

print("Repository structure verified.")

In [ ]:
# load google trend data

if GT_FILE.exists():
    google = pd.read_csv(GT_FILE)
    print(f"Loaded: {GT_FILE.name}")
else:
    if not GT_RAW_FILE.exists():
        raise FileNotFoundError(
            "Google Trends data not found. Expected either "
            f"{GT_FILE.name} or {GT_RAW_FILE.name} in data/raw/."
        )

    google = pd.read_csv(GT_RAW_FILE)
    print(f"Loaded: {GT_RAW_FILE.name}")

In [ ]:
#standarize google data

google["State"] = google["State"].astype(str).str.strip()
google["Year"] = pd.to_numeric(google["Year"], errors="coerce")

google = google[
    google["Year"].between(2013, 2023)
].copy()

google["Year"] = google["Year"].astype(int)

for term in SEARCH_TERMS:
    if term not in google.columns:
        google[term] = np.nan

    google[term] = pd.to_numeric(
        google[term],
        errors="coerce"
    )

if "Search_Term" in google.columns:
    google_year = (
        google
        .groupby(["State", "Year"], as_index=False)[SEARCH_TERMS]
        .mean()
    )
else:
    google_year = google[
        ["State", "Year"] + SEARCH_TERMS
    ].copy()

google_year = (
    google_year
    .sort_values(["State", "Year"])
    .drop_duplicates(["State", "Year"])
    .reset_index(drop=True)
)

print("Google Trends shape:", google_year.shape)
print("States:", google_year["State"].nunique())
print("Years:", sorted(google_year["Year"].unique()))

In [ ]:
# create google trend lag variables

for term in SEARCH_TERMS:
    google_year[f"{term}_lag1"] = (
        google_year
        .groupby("State")[term]
        .shift(1)
    )

google_year.head()

In [ ]:
# load HCUP admission data

if not HCUP_FILE.exists():
    raise FileNotFoundError(
        f"Missing HCUP file: {HCUP_FILE}"
    )

hcup = pd.read_csv(HCUP_FILE)

required_hcup = [
    "State",
    "Year",
    "mental_health_admissions",
]

missing_hcup = [
    column
    for column in required_hcup
    if column not in hcup.columns
]

if missing_hcup:
    raise ValueError(
        f"HCUP dataset is missing: {missing_hcup}"
    )

hcup["State"] = hcup["State"].astype(str).str.strip()
hcup["Year"] = pd.to_numeric(hcup["Year"], errors="coerce")
hcup["mental_health_admissions"] = pd.to_numeric(
    hcup["mental_health_admissions"],
    errors="coerce"
)

hcup = hcup[
    hcup["Year"].between(2013, 2023)
].copy()

hcup["Year"] = hcup["Year"].astype(int)

print("HCUP shape:", hcup.shape)
print("States:", hcup["State"].nunique())
print("Years:", sorted(hcup["Year"].unique()))

In [ ]:
# validate HCUP admissions data

if hcup.duplicated(["State", "Year"]).any():
    raise ValueError("HCUP contains duplicate State-Year rows.")

if hcup["mental_health_admissions"].isna().any():
    raise ValueError("HCUP contains missing admission values.")

if (hcup["mental_health_admissions"] < 0).any():
    raise ValueError("HCUP contains negative admission values.")

print("HCUP validation passed.")

In [ ]:
# load census population data

if not CENSUS_FILE.exists():
    raise FileNotFoundError(
        f"Missing Census file: {CENSUS_FILE}"
    )

census = pd.read_csv(CENSUS_FILE)

required_census = [
    "State",
    "Year",
    "state_population",
]

missing_census = [
    column
    for column in required_census
    if column not in census.columns
]

if missing_census:
    raise ValueError(
        f"Census dataset is missing: {missing_census}"
    )

census["State"] = census["State"].astype(str).str.strip()
census["Year"] = pd.to_numeric(census["Year"], errors="coerce")
census["state_population"] = pd.to_numeric(
    census["state_population"],
    errors="coerce"
)

census = census[
    census["Year"].between(2013, 2023)
].copy()

census = census[
    census["State"].notna()
    & census["state_population"].notna()
].copy()

census["Year"] = census["Year"].astype(int)

census = census[
    ["State", "Year", "state_population"]
].drop_duplicates(
    ["State", "Year"]
)

print("Census shape:", census.shape)
print("States:", census["State"].nunique())

In [ ]:
# validate census data

if census.duplicated(["State", "Year"]).any():
    raise ValueError("Census contains duplicate State-Year rows.")

if (census["state_population"] <= 0).any():
    raise ValueError("Census contains non-positive population values.")

print("Census validation passed.")

In [ ]:
# merge data sets

dataset = (
    google_year
    .merge(
        hcup,
        on=["State", "Year"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        census,
        on=["State", "Year"],
        how="inner",
        validate="one_to_one",
    )
)

print("Merged shape:", dataset.shape)
print("States:", dataset["State"].nunique())
print("Years:", sorted(dataset["Year"].unique()))

In [ ]:
# calculate admission rates

dataset["admission_rate_per_100k"] = (
    dataset["mental_health_admissions"]
    / dataset["state_population"]
    * 100_000
)

dataset["admission_rate_per_100k"].describe()

In [ ]:
# keep predictive modeling period

dataset = dataset[
    dataset["Year"].between(2014, 2023)
].copy()

dataset = dataset.sort_values(
    ["State", "Year"]
).reset_index(drop=True)

print("Final modeling period:", dataset["Year"].min(), "to", dataset["Year"].max())

In [ ]:
#select final variables

lag_columns = [
    f"{term}_lag1"
    for term in SEARCH_TERMS
]

final_columns = [
    "State",
    "Year",
    "state_population",
    "mental_health_admissions",
    "admission_rate_per_100k",
    *lag_columns,
]

dataset = dataset[final_columns].copy()

dataset.head()

In [ ]:
# validate final data set

expected_years = set(range(2014, 2024))

print("=" * 70)
print("FINAL DATASET VALIDATION")
print("=" * 70)

print("\nShape:")
print(dataset.shape)

print("\nStates:")
print(dataset["State"].nunique())

print("\nYears:")
print(sorted(dataset["Year"].unique()))

print("\nDuplicate State-Year rows:")
print(
    dataset.duplicated(
        ["State", "Year"]
    ).sum()
)

print("\nMissing values:")
print(dataset.isna().sum())

print("\nAdmission distribution:")
print(
    dataset["mental_health_admissions"].describe()
)

print("\nAdmission rate distribution:")
print(
    dataset["admission_rate_per_100k"].describe()
)

In [ ]:
#check state-year structure

state_year_check = (
    dataset
    .groupby("State")["Year"]
    .agg(["min", "max", "count"])
)

print(state_year_check)

if dataset.duplicated(["State", "Year"]).any():
    raise ValueError(
        "Final dataset contains duplicate State-Year observations."
    )

if dataset["state_population"].isna().any():
    raise ValueError(
        "Final dataset contains missing population values."
    )

if dataset["mental_health_admissions"].isna().any():
    raise ValueError(
        "Final dataset contains missing outcome values."
    )

if (dataset["state_population"] <= 0).any():
    raise ValueError(
        "Final dataset contains invalid population values."
    )

if (dataset["mental_health_admissions"] < 0).any():
    raise ValueError(
        "Final dataset contains negative admission values."
    )

print("\nFinal validation passed.")

In [ ]:
# check final data set

display(
    dataset
    .sort_values(["State", "Year"])
    .head(25)
)

In [ ]:
# save to processed data

dataset.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {len(dataset.columns)}")

In [ ]:
# confirm saved file available

check = pd.read_csv(OUTPUT_FILE)

print("Saved file:", OUTPUT_FILE)
print("Shape:", check.shape)
print("Duplicate State-Year rows:", check.duplicated(["State", "Year"]).sum())

display(check.head())

In [ ]:
# final summary

print("=" * 70)
print("TEAM RHO DATASET BUILD COMPLETE")
print("=" * 70)

print(f"Output: {OUTPUT_FILE.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {len(dataset.columns)}")
print(f"States: {dataset['State'].nunique()}")
print(f"Years: {dataset['Year'].min()}-{dataset['Year'].max()}")
print(
    f"Duplicate State-Year rows: "
    f"{dataset.duplicated(['State', 'Year']).sum()}"
)
print(
    f"Missing outcome values: "
    f"{dataset['mental_health_admissions'].isna().sum()}"
)
print(
    f"Missing population values: "
    f"{dataset['state_population'].isna().sum()}"
)